# 01 - Data loading

Run this notebook once after cloning the repository. It downloads the three project datasets into the local `data/` directory:

- `data/sroie/`
- `data/cord_v2/`
- `data/resume_parsing_vision/`

The `data/` directory is ignored by Git. Rerunning the notebook reuses completed downloads.

## 1. Install packages

In [1]:
%pip install -q "datasets>=3.0,<5" "ipywidgets>=8,<9"

Note: you may need to restart the kernel to use updated packages.


## 2. Set paths and source versions

The revisions are fixed so every team member downloads the same data.

In [2]:
from pathlib import Path

from datasets import load_dataset, load_from_disk


def is_repo_root(path: Path) -> bool:
    return (path / "README.md").exists() and (path / "notebooks").exists()


def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    candidates = [current, current / "poisoned-paperwork"]
    for parent in current.parents:
        candidates.extend([parent, parent / "poisoned-paperwork"])
    for candidate in candidates:
        if is_repo_root(candidate):
            return candidate
    raise RuntimeError("Could not find the local poisoned-paperwork repository.")


REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data"
CACHE_DIR = DATA_DIR / ".cache" / "huggingface"
DATA_DIR.mkdir(exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

SROIE_ID = "jsdnrs/ICDAR2019-SROIE"
SROIE_REVISION = "bffe40c26759f3376ec2b3ae9031dbba54cd587c"
CORD_ID = "naver-clova-ix/cord-v2"
CORD_REVISION = "7f0115a4b758a71d6473b8d085751692da2fef98"
RESUME_ID = "sukhrobnurali/resume-parsing-vision"
RESUME_REVISION = "3c254be39d45f69c6e20bfb325b4e6ccb3b5e393"

print("Repository:", REPO_ROOT)
print("Data directory:", DATA_DIR)

Repository: /Users/prabudhdkandpal/Desktop/CSCI566/code/poisoned-paperwork
Data directory: /Users/prabudhdkandpal/Desktop/CSCI566/code/poisoned-paperwork/data


## 3. Dataset download helper

In [3]:
def load_or_download(name: str, dataset_id: str, revision: str):
    destination = DATA_DIR / name

    if (destination / "dataset_dict.json").exists():
        print(f"Loading existing {name} data from {destination}")
        return load_from_disk(destination)

    if destination.exists() and any(destination.iterdir()):
        raise RuntimeError(
            f"{destination} contains an incomplete download. "
            "Remove that dataset folder and run this cell again."
        )

    print(f"Downloading {dataset_id}...")
    dataset = load_dataset(
        dataset_id,
        revision=revision,
        cache_dir=str(CACHE_DIR),
    )
    dataset.save_to_disk(destination)
    return dataset

## 4. Download SROIE

In [4]:
sroie = load_or_download("sroie", SROIE_ID, SROIE_REVISION)
print({split: len(data) for split, data in sroie.items()})

Loading existing sroie data from /Users/prabudhdkandpal/Desktop/CSCI566/code/poisoned-paperwork/data/sroie
{'train': 626, 'test': 361}


## 5. Download CORD v2

In [5]:
cord = load_or_download("cord_v2", CORD_ID, CORD_REVISION)
print({split: len(data) for split, data in cord.items()})

Loading existing cord_v2 data from /Users/prabudhdkandpal/Desktop/CSCI566/code/poisoned-paperwork/data/cord_v2
{'train': 800, 'validation': 100, 'test': 100}


## 6. Download English synthetic resumes

This dataset contains 1,000 synthetic English resumes as rendered page images with structured JSON annotations, including `educations[].degree`.

In [6]:
resumes = load_or_download(
    "resume_parsing_vision", RESUME_ID, RESUME_REVISION
)
print({split: len(data) for split, data in resumes.items()})

README.md:   0%|          | 0.00/5.02k [00:00<?, ?B/s]

data/train.parquet: reconstructing file:   0%|          |  0.00B /  279MB            

data/train.parquet: downloading bytes:           |  0.00B            

data/validation.parquet: reconstructing file:   0%|          |  0.00B / 24.4MB            

data/validation.parquet: downloading bytes:           |  0.00B            

...

Generating test split: 0 examples [00:00, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/850 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/75 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/75 [00:00<?, ? examples/s]

{'train': 850, 'validation': 75, 'test': 75}


## 7. Verify the downloads

In [ ]:
counts = {
    "SROIE documents": sum(len(split) for split in sroie.values()),
    "CORD v2 documents": sum(len(split) for split in cord.values()),
    "English synthetic resumes": sum(len(split) for split in resumes.values()),
}
expected = {
    "SROIE documents": 987,
    "CORD v2 documents": 1000,
    "English synthetic resumes": 1000,
}

for name, count in counts.items():
    print(f"{name}: {count:,}")

assert counts == expected, f"Expected {expected}, but found {counts}"
print("\nAll datasets were downloaded successfully.")

SROIE documents: 987
CORD v2 documents: 1,000
English synthetic resumes: 1,000

All datasets were downloaded successfully.


Next, run `02_data_exploration.ipynb` to view example documents and basic dataset statistics.